In [20]:
import torch
from torch import nn, optim

In [21]:
# 自定义模型
class MyModel(nn.Module):
    # 初始化
    def __init__(self):
        super().__init__()
        # 定义一个全连接层
        self.linear = nn.Linear(2, 3)
        # 参数初始化
        self.linear.weight.data = torch.tensor([
            [0.1, 0.2, 0.3],
            [0.4, 0.5, 0.6],
        ]).T
        self.linear.bias.data = torch.tensor([1.0, 2.0, 3.0])
    # 前向传播
    def forward(self, x):
        x = self.linear(x)
        return x

In [22]:
# 1. 准备数据（一条数据样本）
x = torch.tensor([1.0, 2.0])
print(x)
# target = torch.tensor(2)
target = torch.tensor([0, 0, 1]).float()
print(target)

tensor([1., 2.])
tensor([0., 0., 1.])


In [23]:
# 2. 构建模型
model = MyModel()
print(model.linear.weight)
print(model.linear.bias)

Parameter containing:
tensor([[0.1000, 0.4000],
        [0.2000, 0.5000],
        [0.3000, 0.6000]], requires_grad=True)
Parameter containing:
tensor([1., 2., 3.], requires_grad=True)


In [24]:
# 3. 定义损失函数和优化器
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.1)

In [25]:
# 4. 模型训练
# 4.1 前向传播，计算输出值
output = model(x)
print(output)

tensor([1.9000, 3.2000, 4.5000], grad_fn=<ViewBackward0>)


In [26]:
# 4.2 计算损失
loss = loss_fn(output, target)
print(loss)

tensor(0.2977, grad_fn=<DivBackward1>)


In [27]:
# 4.3 反向传播，计算梯度
loss.backward()

In [28]:
print(model.linear.weight.grad)
print(model.linear.bias.grad)

tensor([[ 0.0551,  0.1103],
        [ 0.2024,  0.4047],
        [-0.2575, -0.5150]])
tensor([ 0.0551,  0.2024, -0.2575])


In [29]:
# 4.4 更新参数
optimizer.step()

In [30]:
print(model.linear.weight.data)

tensor([[0.0945, 0.3890],
        [0.1798, 0.4595],
        [0.3258, 0.6515]])


In [31]:
# 4.5 梯度清零
optimizer.zero_grad()

In [32]:
print(model.linear.weight.grad)

None


In [33]:
# 进行下一次迭代
# output = model(x)
# print(output)

In [34]:
# loss = loss_fn(output, target)
# print(loss)

## 梯度计算的验证

In [35]:
print(output)
print(output.argmax(dim=0))

tensor([1.9000, 3.2000, 4.5000], grad_fn=<ViewBackward0>)
tensor(2)


In [36]:
# 计算输出概率
proba = torch.softmax(output, dim=-1)
print(proba)

tensor([0.0551, 0.2024, 0.7425], grad_fn=<SoftmaxBackward0>)


In [37]:
# 计算损失
loss = - target @ torch.log(proba)
print(loss)

tensor(0.2977, grad_fn=<DotBackward0>)


为什么？[Softmax 交叉熵公式总结V2]

In [47]:
# 计算梯度
# 为什么？
print(proba)
print(target)
E = (proba - target).reshape(1, -1)
print(E)
X = x.reshape(1, -1)
print(x,X.T)
grad_W = X.T @ E
print(E)
print(grad_W)

tensor([0.0551, 0.2024, 0.7425], grad_fn=<SoftmaxBackward0>)
tensor([0., 0., 1.])
tensor([[ 0.0551,  0.2024, -0.2575]], grad_fn=<ViewBackward0>)
tensor([1., 2.]) tensor([[1.],
        [2.]])
tensor([[ 0.0551,  0.2024, -0.2575]], grad_fn=<ViewBackward0>)
tensor([[ 0.0551,  0.2024, -0.2575],
        [ 0.1103,  0.4047, -0.5150]], grad_fn=<MmBackward0>)
